In [2]:
import cv2
import numpy as np
import os


def save_step(step_num, title, image, out_dir):
    filename = f"step{step_num}_{title}.png"
    cv2.imwrite(os.path.join(out_dir, filename), image)
    print(f"  Saved: {filename}")


def detect_cracks(image_path: str, output_path: str = "crack_detected.png"):
    out_dir = os.path.dirname(output_path) or "."
    os.makedirs(out_dir, exist_ok=True)

    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    save_step(1, "grayscale", gray, out_dir)

    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(16, 16))
    enhanced = clahe.apply(gray)
    save_step(2, "clahe_enhanced", enhanced, out_dir)

    _, dark_mask = cv2.threshold(enhanced, 60, 255, cv2.THRESH_BINARY_INV)
    save_step(3, "dark_threshold", dark_mask, out_dir)

    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    cleaned = cv2.morphologyEx(dark_mask, cv2.MORPH_OPEN, kernel_open, iterations=1)
    save_step(4, "morph_open_cleaned", cleaned, out_dir)

    kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 20))
    connected = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel_close)
    save_step(5, "morph_close_connected", connected, out_dir)

    contours, _ = cv2.findContours(connected, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    all_contours_img = img.copy()
    cv2.drawContours(all_contours_img, contours, -1, (0, 255, 255), 1)
    save_step(6, "all_contours_before_filter", all_contours_img, out_dir)

    output = img.copy()
    crack_count = 0

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < 200:
            continue

        x, y, w, h = cv2.boundingRect(cnt)
        aspect_ratio = max(w, h) / (min(w, h) + 1e-5)
        hull_area    = cv2.contourArea(cv2.convexHull(cnt))
        solidity     = area / (hull_area + 1e-5)
        perimeter    = cv2.arcLength(cnt, True)
        circularity  = (4 * np.pi * area) / (perimeter ** 2 + 1e-5)

        if area > 5000 and circularity > 0.15 and solidity > 0.7:
            continue

        elif aspect_ratio > 8 and circularity < 0.1 and solidity < 0.65 and max(w, h) > 300:
            cv2.drawContours(output, [cnt], -1, (0, 0, 255), 2)
            cv2.putText(output, "CRACK", (x, y - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)
            crack_count += 1

    save_step(7, "final_result", output, out_dir)

    cv2.imwrite(output_path, output)
    print(f"\nCracks detected: {crack_count}")
    print(f"Final result: {output_path}")
    return output


if __name__ == "__main__":
    detect_cracks(
        image_path="D:\\DIP\\DIP_WORKS\\week 14\\image.png",
        output_path="D:\\DIP\\DIP_WORKS\\week 14\\steps\\crack_detected.png"
    )

  Saved: step1_grayscale.png
  Saved: step2_clahe_enhanced.png
  Saved: step3_dark_threshold.png
  Saved: step4_morph_open_cleaned.png
  Saved: step5_morph_close_connected.png
  Saved: step6_all_contours_before_filter.png
  Saved: step7_final_result.png

Cracks detected: 1
Final result: D:\DIP\DIP_WORKS\week 14\steps\crack_detected.png
